<a href="https://colab.research.google.com/github/gabimazzo-coder/Pre-entrega06/blob/main/Preentrega6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. Instalar duckdb !pip install duckdb --quiet

import duckdb

# 2. Crear una conexión en memoria de DuckDB
con = duckdb.connect(database=':memory:')

# 3. Defino la URL del archivo
url_archivo = "https://ciam.ambiente.gob.ar/dt_csv.php?dt_id=623"

# 4. Registro el archivo como una vista/tabla virtual utilizando la URL directamente
# DuckDB detecta automáticamente el format
con.execute(f"CREATE VIEW tabla_virtual AS SELECT * FROM read_csv_auto('{url_archivo}', delim=';', strict_mode=false, ignore_errors=true)")

# 5. Pruebo que la tabla virtual funciona realizando una consulta
print("Primeros 5 registros de la tabla virtual:")
df_resultado = con.execute("SELECT * FROM tabla_virtual LIMIT 5").df()

# Mostrar el resultado en el formato nativo de Colab
df_resultado

Primeros 5 registros de la tabla virtual:


,estación_meteorológica,año_2010,año_2011,año_2012,año_2013,año_2014,año_2015,año_2016,año_2017,año_2018,año_2019,año_2020,año_2021,año_2022,año_2023,año_2024
0,Buenos Aires,1349,926,1638,1146,1986,1157,1224,1187,1473,1208,834,958,752,952,1162
1,Catamarca,248,359,251,455,555,599,440,512,441,561,401,415,493,505,267
2,Córdoba,793,574,787,781,865,1053,988,682,606,921,684,756,597,516,810
3,Corrientes,1286,1270,1041,1459,1458,1478,1677,1613,1709,1715,879,1027,822,1221,1756
4,Formosa,1303,1248,1205,1349,1621,1864,1447,1379,1500,1351,1102,1311,1086,1506,1258


In [4]:
# 1. Instalar duckdb !pip install duckdb --quiet

import duckdb

# 2. Crear una conexión en memoria de DuckDB
con = duckdb.connect(database=':memory:')

# 3. Defino la URL del archivo url_archivo = "https://ciam.ambiente.gob.ar/dt_csv.php?dt_id=623"

# 4. Registro el archivo como una vista/tabla virtual utilizando la URL directamente
# DuckDB detectao puedes usar funciones específicas como read_csv_auto
con.execute(f"CREATE VIEW tabla_virtual AS SELECT * FROM read_csv_auto('{url_archivo}', delim=';', strict_mode=false, ignore_errors=true)")

# 5. Pruebo que la tabla virtual funciona realizando una consulta
print("Primeros 5 registros de la tabla virtual:")
df_resultado = con.execute("SELECT * FROM tabla_virtual LIMIT 5").df()

# Muestro el resultado en el formato nativo de Colab
df_resultado

# 5.1 Crea una tabla a partir de la vista para poder realizar operaciones de UPDATE
con.execute("CREATE TABLE temperaturas AS SELECT * FROM tabla_virtual")

# 5.2 Limpieza básica: Manejo de nulos
# Tomando la columna se llama 'año_2010' (o la columna de año que corresponda)
# Reemplazaremos los valores NULL con el promedio de esa misma columna
columna_temperatura = "año_2010" # Reemplazar 'Mean' por un nombre de columna existente, por ejemplo 'año_2010'

con.execute(f"""
    UPDATE temperaturas
    SET {columna_temperatura} = (SELECT AVG({columna_temperatura}) FROM temperaturas)
    WHERE {columna_temperatura} IS NULL
""")

# 6. Verificar el resultado
print("\n--- Conteo de valores nulos después de la limpieza ---")
nulos = con.execute(f"SELECT COUNT(*) FROM temperaturas WHERE {columna_temperatura} IS NULL").fetchone()[0]
print(f"Valores nulos restantes en '{columna_temperatura}': {nulos}")

# 7. Exportar el resultado limpio a un DataFrame de Pandas si lo necesitas
df_limpio = con.execute("SELECT * FROM temperaturas").fetchdf()

Primeros 5 registros de la tabla virtual:

--- Conteo de valores nulos después de la limpieza ---
Valores nulos restantes en 'año_2010': 0


In [5]:
# 1. Instalar y e importar las librerías necesarias
!pip install duckdb pandas -q

import duckdb
import pandas as pd

# 2. Configurar la URL del dataset (Reemplaza con tu URL real)
# Nota: DuckDB puede leer directamente formatos CSV, Parquet, JSON, etc., desde la Web.
url_dataset = "https://ciam.ambiente.gob.ar/dt_csv.php?dt_id=623"

# Conectar a una base de datos DuckDB en memoria
con = duckdb.connect(database=':memory:')

# 3. Vista previa de los datos para entender las columnas
print("--- Vista previa de los datos ---")
# FIX: Use read_csv_auto with correct parameters to read the CSV from URL
preview = con.execute(f"SELECT * FROM read_csv_auto('{url_dataset}', delim=';', strict_mode=false, ignore_errors=true) LIMIT 5").df()
print(preview)

# 4. Análisis Agregado 1: Promedios anuales por estación (corrected from 'mensual' due to data structure)
# NOTA: Ajusta los nombres de las columnas ('estación_meteorológica', 'año_2010') según tu dataset.
print("\n--- Promedios Anuales por Estación ---")
query_anual_promedio_estacion = f"""
    SELECT
        "estación_meteorológica",
        AVG("año_2010") AS promedio_año_2010 -- Using 'año_2010' as an example year
    FROM read_csv_auto('{url_dataset}', delim=';', strict_mode=false, ignore_errors=true)
    GROUP BY "estación_meteorológica"
    ORDER BY "estación_meteorológica"
"""
df_anual_promedio_estacion = con.execute(query_anual_promedio_estacion).df()
print(df_anual_promedio_estacion.head(10))

# 5. Análisis Agregado 2: Máximos históricos por estación / región
# NOTA: Ajusta 'estación_meteorológica' y el año de la columna según tu dataset.
print("\n--- Máximos Históricos por Estación/Región (Año 2010) ---")
query_maximos = f"""
    SELECT
        "estación_meteorológica" AS estacion_region,
        MAX("año_2010") AS maximo_historico_año_2010 -- Using 'año_2010' as an example year
    FROM read_csv_auto('{url_dataset}', delim=';', strict_mode=false, ignore_errors=true)
    WHERE "estación_meteorológica" IS NOT NULL
    GROUP BY estacion_region
    ORDER BY maximo_historico_año_2010 DESC
    LIMIT 10
"""
df_maximos = con.execute(query_maximos).df()
print(df_maximos)

--- Vista previa de los datos ---
  estación_meteorológica  año_2010  año_2011  año_2012  año_2013  año_2014  \
0          Buenos Aires       1349       926      1638      1146      1986   
1             Catamarca        248       359       251       455       555   
2               Córdoba        793       574       787       781       865   
3            Corrientes       1286      1270      1041      1459      1458   
4                Formosa      1303      1248      1205      1349      1621   

   año_2015  año_2016  año_2017  año_2018  año_2019  año_2020  año_2021  \
0      1157      1224      1187      1473      1208       834       958   
1       599       440       512       441       561       401       415   
2      1053       988       682       606       921       684       756   
3      1478      1677      1613      1709      1715       879      1027   
4      1864      1447      1379      1500      1351      1102      1311   

   año_2022  año_2023  año_2024  
0       752 